In [1]:
import os
os.chdir(r"C:\Users\HP\Desktop\image-search")
print(os.getcwd())

C:\Users\HP\Desktop\image-search


In [2]:
import sys
sys.path.append(".")

In [3]:
import aiohttp
import numpy as np
from tqdm import tqdm
from src.utils import load_image_from_url
from models.embedder import embed_images_batch
import asyncio

c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
import pandas as pd

df = pd.read_csv(
    "data/unsplash_lite/photos.csv",
    sep="\t",
    on_bad_lines="skip"
)

print(len(df))
print(df.columns)

25000
Index(['photo_id', 'photo_url', 'photo_image_url', 'photo_submitted_at',
       'photo_featured', 'photo_width', 'photo_height', 'photo_aspect_ratio',
       'photo_description', 'photographer_username', 'photographer_first_name',
       'photographer_last_name', 'exif_camera_make', 'exif_camera_model',
       'exif_iso', 'exif_aperture_value', 'exif_focal_length',
       'exif_exposure_time', 'photo_location_name', 'photo_location_latitude',
       'photo_location_longitude', 'photo_location_country',
       'photo_location_city', 'stats_views', 'stats_downloads',
       'ai_description', 'ai_primary_landmark_name',
       'ai_primary_landmark_latitude', 'ai_primary_landmark_longitude',
       'ai_primary_landmark_confidence', 'blur_hash'],
      dtype='object')


In [5]:
df = df.dropna(subset=["photo_image_url"])
df = df[df["photo_image_url"].str.startswith("https")]
df = df.reset_index(drop=True)

print("Total usable:", len(df))

Total usable: 25000


In [6]:
LIMIT = 10000
df = df.iloc[:LIMIT]

In [7]:
async def run_batched(df, limit=10000, batch_size=32):

    embeddings = []
    ids = []

    connector = aiohttp.TCPConnector(limit=20)
    timeout = aiohttp.ClientTimeout(total=10)

    df = df.iloc[:limit]

    batch_images = []
    batch_ids = []

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:

        for _, row in tqdm(df.iterrows(), total=len(df)):

            try:
                img = await load_image_from_url(session, row["photo_image_url"])
            except:
                continue

            if img is None:
                continue

            batch_images.append(img)
            batch_ids.append(row["photo_id"])

            if len(batch_images) >= batch_size:
                vecs = embed_images_batch(batch_images)

                embeddings.append(vecs)
                ids.extend(batch_ids)

                batch_images.clear()
                batch_ids.clear()

        # flush remaining
        if batch_images:
            vecs = embed_images_batch(batch_images)
            embeddings.append(vecs)
            ids.extend(batch_ids)

    embeddings = np.concatenate(embeddings, axis=0).astype("float32")

    return ids, embeddings

In [ ]:
ids, embeddings = await run_batched(df, batch_size=32)

 66%|██████▋   | 665/1000 [41:30<05:43,  1.03s/it]  c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
100%|██████████| 1000/1000 [57:53<00:00,  3.47s/it]


In [10]:
print(embeddings.shape, len(ids))

# NaN check
print("NaNs:", np.isnan(embeddings).any())

# Norms
norms = np.linalg.norm(embeddings, axis=1)
print(norms[:10])
print("Min norm:", norms.min(), "Max norm:", norms.max())

(993, 768) 993
NaNs: False
[1.         1.         1.         1.         1.         0.99999994
 1.         0.99999994 1.         1.        ]
Min norm: 0.9999998 Max norm: 1.0000001


In [11]:
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

In [12]:
import faiss

d = embeddings.shape[1]

index = faiss.IndexFlatIP(d)  # cosine similarity (after normalization)
index.add(embeddings)

print("Index size:", index.ntotal)

Index size: 993


In [13]:
import random

i = random.randint(0, len(ids)-1)

query_vec = embeddings[i].reshape(1, -1)

distances, indices = index.search(query_vec, k=5)

print(indices)
print(distances)

[[ 55 789  35 467 972]]
[[0.9999999  0.94283164 0.9353559  0.9302403  0.9288718 ]]


In [14]:
results = [ids[idx] for idx in indices[0]]
print("Query ID:", ids[i])
print("Results:", results)

Query ID: 4k-U1Wp2d00
Results: ['4k-U1Wp2d00', 'kXYmB2m-poo', '4qZq7xlqpvM', 'xVwo6NxKDFk', '4CDdd1RCt6w']


In [16]:
np.save("data/unsplash/embeddings.npy", embeddings)
np.save("data/unsplash/ids.npy", np.array(ids))
faiss.write_index(index, "data/unsplash/index.faiss")

In [17]:
def search_by_index(i, k=5):
    query = embeddings[i].reshape(1, -1)
    distances, indices = index.search(query, k)

    print("Query ID:", ids[i])
    print("Results:", [ids[idx] for idx in indices[0]])
    print("Scores:", distances[0])

In [18]:
search_by_index(0)
search_by_index(50)
search_by_index(200)

Query ID: oSf8ePoG9NU
Results: ['oSf8ePoG9NU', 'Enr71dsAO5w', '-N_UwPdUs7E', 'qzj2SNRNqeY', 'TwMLYN_CMT4']
Scores: [1.         0.8588593  0.85090554 0.848215   0.8476877 ]
Query ID: F1OjCxOlj8Q
Results: ['F1OjCxOlj8Q', 'Py_ofnAnc-E', 'pdzQ1cAdftk', 'jEoAMd9K5EE', 'WlVZSJVLLQg']
Scores: [1.         0.874503   0.8216782  0.803404   0.80309916]
Query ID: I3nHRNu3960
Results: ['I3nHRNu3960', 'savOaIMns3Y', 'eyPiErJ6XcI', 'Z-JcRkLAAkk', 'ZMEmMguGD1I']
Scores: [1.0000001  0.84926885 0.8235841  0.8163389  0.81143224]


In [19]:
search_by_index(100)
search_by_index(101)
search_by_index(102)

Query ID: _ppnPXy_TVw
Results: ['_ppnPXy_TVw', 'nmg5w3EyAFM', 'qY1NKI6RUwQ', '2sT1wre6vfU', 'kU9NnI73MRg']
Scores: [0.9999999  0.8965961  0.87381494 0.8724127  0.8631531 ]
Query ID: htAyoWqrXww
Results: ['htAyoWqrXww', 'cOzOIsKvsUQ', 'D2Q2o7bYWew', 'ESus7wfHOas', 'KQNBuD9YGdo']
Scores: [1.         0.8922336  0.8682233  0.8670151  0.84652495]
Query ID: ka8xTk3KMw0
Results: ['ka8xTk3KMw0', 'JIF__24w3XA', 'EmSpZYr3hEg', 'FjikPptEbZg', 'gmHVdAGe2EI']
Scores: [1.         0.90591204 0.89874387 0.8987119  0.8820751 ]


In [20]:
import random

for _ in range(5):
    search_by_index(random.randint(0, 999))

Query ID: CF33GT7iM9c
Results: ['CF33GT7iM9c', 'Ls0oRZd-NuA', 'fhiP58dYuFU', 'EIeN8My2vLY', 'ftloE4xFjVQ']
Scores: [1.         0.792855   0.78981507 0.7811877  0.77954185]
Query ID: x9RN-ZIzkf0
Results: ['x9RN-ZIzkf0', 'sfsvTLQh0o4', 'cv_fqvrbKWs', '_zHYUQmWrzk', 'Fnb73SxGQCk']
Scores: [1.0000001  0.9392633  0.9351405  0.92425936 0.92074156]
Query ID: Xne1N4yZuOY
Results: ['Xne1N4yZuOY', 'Enr71dsAO5w', 'zXEme0luaxk', 'QPg8RkHRIOo', 'e0qi7y5348o']
Scores: [1.         0.7752305  0.76381433 0.7632276  0.7593982 ]
Query ID: o51TCXsqCh4
Results: ['o51TCXsqCh4', 'Nk3-4mE6h2U', 'jgV1dF5Hb4A', '5ErbZB4VY3M', 'Yh6K2eTr_FY']
Scores: [1.0000001  0.95811707 0.9459145  0.9204657  0.9117894 ]
Query ID: rAP497AwNoU
Results: ['rAP497AwNoU', 'RNrh6mIrOxo', 'Sk1x2a-xKKY', 'S5RBgaziupg', 'z2gSHd515EY']
Scores: [1.         0.9200413  0.9111876  0.90474236 0.89628285]
